# 🔥 Ironhaven Safety Intelligence — Predictive Calamity Analysis

**Full-stack AI agent for underground tunnel safety monitoring**

This notebook performs:
1. **Exploratory Data Analysis** — sensor distributions, correlations, temporal patterns
2. **Feature Engineering** — rolling statistics, rate-of-change, cross-sensor interaction terms
3. **Model Training** — XGBoost, Random Forest, and LSTM compared head-to-head
4. **SHAP Explainability** — why the model flags each reading
5. **Predictive Analysis** — explosion-risk scoring and pattern identification
6. **Model Export** — serialized model ready for the FastAPI backend

---

## 1. Setup & Imports

In [ ]:
# Install required packages
!pip install -q xgboost shap scikit-learn matplotlib seaborn tensorflow pandas numpy joblib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    f1_score, precision_score, recall_score, roc_auc_score
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
import xgboost as xgb
import shap
import joblib
import warnings
warnings.filterwarnings('ignore')

# Visualization config
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
COLORS = {'Normal': '#22c55e', 'Warning': '#f59e0b', 'Danger': '#ef4444'}

print('✅ All imports ready')

## 2. Data Loading & Initial Inspection

In [ ]:
# Upload dataset
from google.colab import files
uploaded = files.upload()  # Upload tunnel_monitoring_dataset_5000.csv

filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df = df.sort_values('Timestamp').reset_index(drop=True)

print(f'Dataset: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Time range: {df.Timestamp.min()} → {df.Timestamp.max()}')
print(f'Sampling interval: {(df.Timestamp.diff().median())} (median)')
print(f'\nNull values: {df.isnull().sum().sum()}')
df.head(10)

In [ ]:
# Class distribution
status_counts = df['Status'].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
bars = axes[0].bar(status_counts.index, status_counts.values,
                   color=[COLORS[s] for s in status_counts.index], edgecolor='white', linewidth=2)
axes[0].set_title('Status Distribution', fontweight='bold', fontsize=14)
axes[0].set_ylabel('Count')
for bar, val in zip(bars, status_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                f'{val} ({val/len(df)*100:.1f}%)', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(status_counts.values, labels=status_counts.index,
            colors=[COLORS[s] for s in status_counts.index],
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Status Proportions', fontweight='bold', fontsize=14)

plt.tight_layout()
plt.show()
print(f'⚠️ Class imbalance: Normal is only {status_counts.get("Normal",0)/len(df)*100:.1f}% of data')

## 3. Exploratory Data Analysis

### 3.1 Sensor Distributions by Status

In [ ]:
sensor_cols = ['Temperature', 'Humidity', 'CO_Level', 'CO2_Level',
               'Methane', 'Smoke_Level', 'AQI', 'Light_Intensity', 'Vibration']

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
for idx, col in enumerate(sensor_cols):
    ax = axes[idx // 3][idx % 3]
    for status in ['Normal', 'Warning', 'Danger']:
        subset = df[df['Status'] == status][col]
        ax.hist(subset, bins=40, alpha=0.5, label=status, color=COLORS[status], density=True)
    ax.set_title(col, fontweight='bold')
    ax.legend(fontsize=8)
    ax.set_ylabel('Density')

plt.suptitle('Sensor Distributions by Status', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 3.2 Correlation Heatmap

In [ ]:
# Correlation matrix
le_temp = LabelEncoder()
df['Status_Encoded'] = le_temp.fit_transform(df['Status'])

corr_cols = sensor_cols + ['Status_Encoded']
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, square=True, linewidths=1, ax=ax,
            cbar_kws={'label': 'Correlation'})
ax.set_title('Sensor Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n🔑 Top correlators with Status:')
status_corr = corr_matrix['Status_Encoded'].drop('Status_Encoded').abs().sort_values(ascending=False)
for feat, val in status_corr.items():
    print(f'  {feat:20s} {val:.4f}')

### 3.3 Temporal Patterns

In [ ]:
# Time-series view of key sensors
fig, axes = plt.subplots(4, 1, figsize=(18, 16), sharex=True)
color_map = df['Status'].map(COLORS)

for ax, col in zip(axes, ['CO_Level', 'Methane', 'Smoke_Level', 'Temperature']):
    ax.scatter(df['Timestamp'], df[col], c=color_map, alpha=0.4, s=3)
    ax.set_ylabel(col, fontweight='bold')
    ax.grid(True, alpha=0.3)

axes[0].set_title('Temporal Sensor Readings (colored by Status)', fontsize=14, fontweight='bold')
axes[-1].set_xlabel('Timestamp')

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=s) for s, c in COLORS.items()]
axes[0].legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
plt.show()

### 3.4 Danger Condition Pattern Analysis

In [ ]:
# Identify explosive risk conditions:
# High heat + high methane + high vibration = explosion risk
# High CO + high smoke = fire risk

df['explosion_risk_score'] = (
    MinMaxScaler().fit_transform(df[['Temperature']]) * 0.3 +
    MinMaxScaler().fit_transform(df[['Methane']]) * 0.4 +
    MinMaxScaler().fit_transform(df[['Vibration']]) * 0.3
).flatten()

df['fire_risk_score'] = (
    MinMaxScaler().fit_transform(df[['CO_Level']]) * 0.5 +
    MinMaxScaler().fit_transform(df[['Smoke_Level']]) * 0.5
).flatten()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, risk, title in zip(axes,
    ['explosion_risk_score', 'fire_risk_score'],
    ['Explosion Risk (Temp+Methane+Vibration)', 'Fire Risk (CO+Smoke)']):
    for status in ['Normal', 'Warning', 'Danger']:
        subset = df[df['Status'] == status]
        ax.hist(subset[risk], bins=40, alpha=0.5, label=status,
                color=COLORS[status], density=True)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Risk Score')
    ax.legend()

plt.tight_layout()
plt.show()

# Cross-analysis: CO vs Methane colored by status
fig, ax = plt.subplots(figsize=(10, 8))
for status in ['Normal', 'Warning', 'Danger']:
    subset = df[df['Status'] == status]
    ax.scatter(subset['CO_Level'], subset['Methane'], alpha=0.4,
              color=COLORS[status], label=status, s=15)
ax.set_xlabel('CO Level (ppm)', fontweight='bold')
ax.set_ylabel('Methane (%LEL)', fontweight='bold')
ax.set_title('CO vs Methane — Danger Zone Mapping', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Feature Engineering

In [ ]:
# ====== Temporal features ======
df['hour'] = df['Timestamp'].dt.hour
df['day_of_week'] = df['Timestamp'].dt.dayofweek
df['is_night'] = ((df['hour'] >= 22) | (df['hour'] <= 5)).astype(int)

# ====== Rolling statistics (12-window = 1 hour) ======
window = 12
for col in sensor_cols:
    df[f'{col}_rolling_mean'] = df[col].rolling(window, min_periods=1).mean()
    df[f'{col}_rolling_std'] = df[col].rolling(window, min_periods=1).std().fillna(0)
    df[f'{col}_rate_of_change'] = df[col].diff().fillna(0)

# ====== Cross-sensor interaction features ======
df['CO_x_Methane'] = df['CO_Level'] * df['Methane']
df['CO_x_Smoke'] = df['CO_Level'] * df['Smoke_Level']
df['Temp_x_Methane'] = df['Temperature'] * df['Methane']
df['Temp_x_Vibration'] = df['Temperature'] * df['Vibration']
df['Methane_x_Vibration'] = df['Methane'] * df['Vibration']
df['AQI_x_Smoke'] = df['AQI'] * df['Smoke_Level']

# ====== Lag features (look-back for temporal patterns) ======
for lag in [1, 3, 6]:
    for col in ['CO_Level', 'Methane', 'Smoke_Level', 'Temperature']:
        df[f'{col}_lag{lag}'] = df[col].shift(lag).fillna(method='bfill')

# ====== Composite risk scores (already computed) ======
# explosion_risk_score and fire_risk_score from EDA

print(f'Features after engineering: {df.shape[1]} columns')

# List all features
feature_cols = [c for c in df.columns if c not in ['Timestamp', 'Status', 'Status_Encoded']]
print(f'\nFeature count: {len(feature_cols)}')
print(f'Features: {feature_cols}')

## 5. Model Training — Three-Way Comparison

### 5.1 Data Preparation

In [ ]:
# Encode labels
le = LabelEncoder()
y = le.fit_transform(df['Status'])  # Danger=0, Normal=1, Warning=2
print('Label mapping:', dict(zip(le.classes_, le.transform(le.classes_))))

X = df[feature_cols].values

# Train/test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale for RF and LSTM
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Class weights for imbalance
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
cw_dict = dict(zip(np.unique(y_train), class_weights))
print(f'\nClass weights: {cw_dict}')
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

### 5.2 XGBoost

In [ ]:
# XGBoost with class weights
sample_weights_train = np.array([cw_dict[yi] for yi in y_train])

xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    eval_metric='mlogloss',
    use_label_encoder=False
)

xgb_model.fit(X_train, y_train, sample_weight=sample_weights_train,
              eval_set=[(X_test, y_test)], verbose=False)

y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)

print('=== XGBoost Results ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_xgb):.4f}')
print(f'Macro F1: {f1_score(y_test, y_pred_xgb, average="macro"):.4f}')
print(f'Weighted F1: {f1_score(y_test, y_pred_xgb, average="weighted"):.4f}')
print()
print(classification_report(y_test, y_pred_xgb, target_names=le.classes_))

### 5.3 Random Forest

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_scaled, y_train)
y_pred_rf = rf_model.predict(X_test_scaled)
y_prob_rf = rf_model.predict_proba(X_test_scaled)

print('=== Random Forest Results ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}')
print(f'Macro F1: {f1_score(y_test, y_pred_rf, average="macro"):.4f}')
print(f'Weighted F1: {f1_score(y_test, y_pred_rf, average="weighted"):.4f}')
print()
print(classification_report(y_test, y_pred_rf, target_names=le.classes_))

### 5.4 LSTM (Deep Learning)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

# Reshape for LSTM: (samples, timesteps, features)
# Use sequences of 6 time steps (30 min lookback)
SEQ_LEN = 6

def create_sequences(X, y, seq_len):
    Xs, ys = [], []
    for i in range(seq_len, len(X)):
        Xs.append(X[i-seq_len:i])
        ys.append(y[i])
    return np.array(Xs), np.array(ys)

# Use the full sorted dataset for sequences (scale first)
X_all_scaled = scaler.fit_transform(df[feature_cols].values)
y_all = le.transform(df['Status'].values)

X_seq, y_seq = create_sequences(X_all_scaled, y_all, SEQ_LEN)

# Split
split_idx = int(len(X_seq) * 0.8)
X_train_seq, X_test_seq = X_seq[:split_idx], X_seq[split_idx:]
y_train_seq, y_test_seq = y_seq[:split_idx], y_seq[split_idx:]

y_train_cat = to_categorical(y_train_seq, num_classes=3)
y_test_cat = to_categorical(y_test_seq, num_classes=3)

print(f'LSTM sequences — Train: {X_train_seq.shape} | Test: {X_test_seq.shape}')

# Build LSTM
lstm_model = Sequential([
    LSTM(128, return_sequences=True, input_shape=(SEQ_LEN, X_train_seq.shape[2])),
    Dropout(0.3),
    BatchNormalization(),
    LSTM(64, return_sequences=False),
    Dropout(0.3),
    BatchNormalization(),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(3, activation='softmax')
])

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Compute class weights for LSTM
cw_lstm = compute_class_weight('balanced', classes=np.unique(y_train_seq), y=y_train_seq)
cw_lstm_dict = dict(zip(np.unique(y_train_seq), cw_lstm))

callbacks = [
    EarlyStopping(patience=10, restore_best_weights=True, monitor='val_loss'),
    ReduceLROnPlateau(factor=0.5, patience=5, min_lr=1e-6)
]

history = lstm_model.fit(
    X_train_seq, y_train_cat,
    validation_data=(X_test_seq, y_test_cat),
    epochs=100, batch_size=64,
    class_weight=cw_lstm_dict,
    callbacks=callbacks, verbose=1
)

In [ ]:
# LSTM evaluation
y_pred_lstm_prob = lstm_model.predict(X_test_seq)
y_pred_lstm = np.argmax(y_pred_lstm_prob, axis=1)

print('=== LSTM Results ===')
print(f'Accuracy: {accuracy_score(y_test_seq, y_pred_lstm):.4f}')
print(f'Macro F1: {f1_score(y_test_seq, y_pred_lstm, average="macro"):.4f}')
print(f'Weighted F1: {f1_score(y_test_seq, y_pred_lstm, average="weighted"):.4f}')
print()
print(classification_report(y_test_seq, y_pred_lstm, target_names=le.classes_))

## 6. Head-to-Head Model Comparison

In [ ]:
# Comparison table
results = {
    'XGBoost': {
        'Accuracy': accuracy_score(y_test, y_pred_xgb),
        'Macro F1': f1_score(y_test, y_pred_xgb, average='macro'),
        'Weighted F1': f1_score(y_test, y_pred_xgb, average='weighted'),
        'Macro Precision': precision_score(y_test, y_pred_xgb, average='macro'),
        'Macro Recall': recall_score(y_test, y_pred_xgb, average='macro'),
    },
    'Random Forest': {
        'Accuracy': accuracy_score(y_test, y_pred_rf),
        'Macro F1': f1_score(y_test, y_pred_rf, average='macro'),
        'Weighted F1': f1_score(y_test, y_pred_rf, average='weighted'),
        'Macro Precision': precision_score(y_test, y_pred_rf, average='macro'),
        'Macro Recall': recall_score(y_test, y_pred_rf, average='macro'),
    },
    'LSTM': {
        'Accuracy': accuracy_score(y_test_seq, y_pred_lstm),
        'Macro F1': f1_score(y_test_seq, y_pred_lstm, average='macro'),
        'Weighted F1': f1_score(y_test_seq, y_pred_lstm, average='weighted'),
        'Macro Precision': precision_score(y_test_seq, y_pred_lstm, average='macro'),
        'Macro Recall': recall_score(y_test_seq, y_pred_lstm, average='macro'),
    }
}

results_df = pd.DataFrame(results).T
print('\n📊 MODEL COMPARISON')
print('='*70)
print(results_df.round(4).to_string())
print('='*70)

best_model = results_df['Macro F1'].idxmax()
print(f'\n🏆 Best model by Macro F1: {best_model} ({results_df.loc[best_model, "Macro F1"]:.4f})')

In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar comparison
metrics = ['Accuracy', 'Macro F1', 'Weighted F1']
x = np.arange(len(metrics))
width = 0.25
model_colors = ['#3b82f6', '#10b981', '#8b5cf6']

for i, (model, color) in enumerate(zip(results_df.index, model_colors)):
    vals = [results_df.loc[model, m] for m in metrics]
    axes[0].bar(x + i*width, vals, width, label=model, color=color, edgecolor='white')

axes[0].set_xlabel('Metric')
axes[0].set_ylabel('Score')
axes[0].set_title('Model Performance Comparison', fontweight='bold')
axes[0].set_xticks(x + width)
axes[0].set_xticklabels(metrics)
axes[0].legend()
axes[0].set_ylim(0, 1.1)

# Confusion matrices
from matplotlib.gridspec import GridSpec
fig2, axes2 = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, y_true, y_pred) in zip(axes2, [
    ('XGBoost', y_test, y_pred_xgb),
    ('Random Forest', y_test, y_pred_rf),
    ('LSTM', y_test_seq, y_pred_lstm)
]):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=le.classes_, yticklabels=le.classes_)
    ax.set_title(f'{name}', fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. SHAP Explainability (XGBoost)

In [ ]:
# SHAP analysis on the best tree model
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

# Feature names
feature_names = feature_cols

# Summary plot — Danger class
print('🔍 SHAP Feature Importance — what drives DANGER predictions:')
danger_idx = list(le.classes_).index('Danger')

fig, ax = plt.subplots(figsize=(12, 10))
shap.summary_plot(shap_values[danger_idx], X_test,
                  feature_names=feature_names, show=False, max_display=20)
plt.title('SHAP Values — Danger Class', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Top 15 features by mean absolute SHAP value
mean_shap = np.abs(shap_values[danger_idx]).mean(axis=0)
feat_importance = pd.DataFrame({
    'Feature': feature_names,
    'Mean |SHAP|': mean_shap
}).sort_values('Mean |SHAP|', ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))
top15 = feat_importance.head(15)
ax.barh(top15['Feature'][::-1], top15['Mean |SHAP|'][::-1], color='#ef4444', edgecolor='white')
ax.set_xlabel('Mean |SHAP Value|', fontweight='bold')
ax.set_title('Top 15 Features Driving Danger Predictions', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

print('\n📋 Full feature ranking:')
print(feat_importance.head(20).to_string(index=False))

## 8. Predictive Analysis — AI Agent Insights

In [ ]:
# ====== CALAMITY PREDICTION ENGINE ======
# Analyze what conditions lead to Danger and identify critical thresholds

danger_df = df[df['Status'] == 'Danger']
normal_df = df[df['Status'] == 'Normal']

print('🚨 CALAMITY CONDITION ANALYSIS')
print('='*60)
print(f'\nDanger readings: {len(danger_df)} ({len(danger_df)/len(df)*100:.1f}%)')
print(f'Normal readings: {len(normal_df)} ({len(normal_df)/len(df)*100:.1f}%)')

print('\n--- Critical Thresholds (Danger vs Normal mean) ---')
for col in sensor_cols:
    d_mean = danger_df[col].mean()
    n_mean = normal_df[col].mean()
    ratio = d_mean / n_mean if n_mean != 0 else float('inf')
    print(f'  {col:20s}  Danger avg: {d_mean:8.2f}  Normal avg: {n_mean:8.2f}  Ratio: {ratio:.2f}x')

# Identify extreme danger windows (consecutive danger readings)
df['danger_flag'] = (df['Status'] == 'Danger').astype(int)
df['danger_streak'] = df['danger_flag'].groupby(
    (df['danger_flag'] != df['danger_flag'].shift()).cumsum()
).cumcount() + 1
df.loc[df['danger_flag'] == 0, 'danger_streak'] = 0

max_streak = df['danger_streak'].max()
print(f'\n⚠️  Longest consecutive Danger streak: {max_streak} readings ({max_streak*5} minutes)')

# High-risk windows (streak >= 5 = 25+ minutes of danger)
high_risk = df[df['danger_streak'] >= 5]
print(f'High-risk sustained danger readings (5+ consecutive): {len(high_risk)}')

In [ ]:
# ====== DECISION BOUNDARIES ======
# Find optimal thresholds for each key sensor

from sklearn.tree import DecisionTreeClassifier

# Simple decision tree to extract human-readable rules
dt = DecisionTreeClassifier(max_depth=4, class_weight='balanced', random_state=42)
dt.fit(df[sensor_cols].values, le.transform(df['Status']))

from sklearn.tree import export_text
rules = export_text(dt, feature_names=sensor_cols, max_depth=4)
print('🌳 Decision Rules for Status Classification:')
print(rules)

# Critical threshold summary
print('\n' + '='*60)
print('🔑 KEY FINDINGS — AI AGENT RULES')
print('='*60)
print(f'''
EXPLOSION RISK CONDITIONS:
  • CO Level > {danger_df["CO_Level"].quantile(0.75):.1f} ppm
  • Methane > {danger_df["Methane"].quantile(0.75):.1f} %LEL
  • Temperature > {danger_df["Temperature"].quantile(0.75):.1f}°C
  • Combined: CO × Methane > {danger_df["CO_x_Methane"].quantile(0.75):.1f}

FIRE RISK CONDITIONS:
  • CO Level > {danger_df["CO_Level"].quantile(0.9):.1f} ppm
  • Smoke Level > {danger_df["Smoke_Level"].quantile(0.75):.1f}
  • AQI > {danger_df["AQI"].quantile(0.75):.1f}

STRUCTURAL RISK:
  • Vibration > {danger_df["Vibration"].quantile(0.9):.3f} g
  • Sustained danger streak > {int(max_streak * 0.5)} readings = evacuate
''')

## 9. Model Export — Ready for Backend Integration

In [ ]:
# Export the best model + scaler + label encoder
import pickle

# Save XGBoost (typically the best for tabular data)
joblib.dump(xgb_model, 'ironhaven_xgb_model.joblib')
joblib.dump(rf_model, 'ironhaven_rf_model.joblib')
joblib.dump(scaler, 'ironhaven_scaler.joblib')
joblib.dump(le, 'ironhaven_label_encoder.joblib')

# Save LSTM
lstm_model.save('ironhaven_lstm_model.keras')

# Save feature list
with open('ironhaven_features.json', 'w') as f:
    import json
    json.dump({
        'feature_cols': feature_cols,
        'sensor_cols': sensor_cols,
        'label_mapping': dict(zip(le.classes_.tolist(), le.transform(le.classes_).tolist())),
        'model_comparison': results_df.to_dict()
    }, f, indent=2)

print('✅ Models exported:')
print('  • ironhaven_xgb_model.joblib')
print('  • ironhaven_rf_model.joblib')
print('  • ironhaven_scaler.joblib')
print('  • ironhaven_label_encoder.joblib')
print('  • ironhaven_lstm_model.keras')
print('  • ironhaven_features.json')

# Download all
from google.colab import files
for f_name in ['ironhaven_xgb_model.joblib', 'ironhaven_rf_model.joblib',
               'ironhaven_scaler.joblib', 'ironhaven_label_encoder.joblib',
               'ironhaven_features.json']:
    files.download(f_name)

print('\n📦 Downloads triggered — save these for the FastAPI backend')

## 10. LSTM Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Train Loss', color='#3b82f6')
axes[0].plot(history.history['val_loss'], label='Val Loss', color='#ef4444')
axes[0].set_title('LSTM Loss Curve', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(history.history['accuracy'], label='Train Accuracy', color='#3b82f6')
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy', color='#ef4444')
axes[1].set_title('LSTM Accuracy Curve', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.show()

## 📋 Summary & Next Steps

### Key Findings
- **CO_Level** and **Methane** are the strongest predictors of Danger status
- Cross-sensor interactions (CO × Methane, CO × Smoke) are highly predictive
- Rolling statistics capture temporal degradation patterns before calamities
- The AI agent can predict danger conditions with the exported model

### Integration Plan
1. **FastAPI microservice** loads the best model and serves predictions via `/predict` endpoint
2. **Node.js backend** calls the FastAPI service on each new sensor reading batch
3. **React dashboard** displays real-time risk scores, SHAP explanations, and alert feed
4. **Alert engine** triggers DGMS-compliant alerts when model predicts Danger

---
*Ironhaven Safety Intelligence — Protecting underground tunnel workers with AI*